In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

# ── S3 ───────────────────────────────────────────────────────────────
AWS_REGION      = "ap-southeast-1"
AWS_BUCKET_NAME = "aic-bucket-2026"

# Ưu tiên Kaggle Secrets (Add-ons -> Secrets). Điền tay vào đây nếu chạy ngoài Kaggle.
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient

    _sec = UserSecretsClient()
    AWS_ACCESS_KEY = AWS_ACCESS_KEY or _sec.get_secret("AWS_ACCESS_KEY")
    AWS_SECRET_KEY = AWS_SECRET_KEY or _sec.get_secret("AWS_SECRET_KEY")
except Exception as ex:
    print(f"! Chưa lấy được Kaggle Secrets ({type(ex).__name__}) — điền tay nếu cần")

S3_URL_BASE = f"https://{AWS_BUCKET_NAME}.s3.{AWS_REGION}.amazonaws.com"

os.environ.update(
    AWS_ACCESS_KEY=AWS_ACCESS_KEY or "",
    AWS_SECRET_KEY=AWS_SECRET_KEY or "",
    AWS_REGION=AWS_REGION,
    AWS_BUCKET_NAME=AWS_BUCKET_NAME,
)

# ── Model SigLIP đã export ONNX (xem services/ingest/src/ingest/export_siglip_onnx.py) ──
SIGLIP_S3_PREFIX = "models/siglip-so400m-patch14-384-onnx"
SIGLIP_LOCAL_DIR = "/kaggle/working/models/siglip-onnx"

# ── Input: keyframes/scenes đã cắt + upload bởi notebook aic-frame-cut-2026 ──
# Cấu trúc trên S3 (do notebook cắt frame ghi ra):
#   Keyframes_{group}/keyframes/{name}/keyframes.json, shots.csv, *.webp
#   Keyscence_{group}/keyscence/{name}/scenes.json, scene_*.mp4
OUT_ROOT   = "/kaggle/working/out"     # nơi tải keyframes/scenes về tạm để embed
EMBED_ROOT = "/kaggle/working/embed"   # nơi ghi embed_*.parquet trước khi upload

# Chia việc theo group, giống notebook cắt frame — chỉ embed group đã cắt xong rồi.
#   người A: GROUPS = ["L21_a", "L22_a", "L23_a"]
#   người B: GROUPS = ["L26_a", "L26_b", "L26_c"]
GROUPS = ["L26_b", "L26_c", "L26_d"]     # [] = tất cả group có trên S3
GROUPS_LAST_N = 0                        # vd 5 = 5 group cuối (theo natural sort), bỏ qua nếu GROUPS đã có giá trị
SHARD, NUM_SHARDS = 0, 1                 # chia nhỏ hơn nữa trong 1 group cho nhiều session

BATCH_SIZE                = 64     # số ảnh/batch khi encode qua ONNX Runtime
DO_UPLOAD                 = True   # True = đẩy embed_*.parquet lên S3 sau mỗi video
DELETE_LOCAL_AFTER_UPLOAD = True   # True = xoá keyframes/scenes đã tải + parquet local sau upload
OVERWRITE                 = False  # True = embed lại cả khi đã có embed_000.parquet trên S3
LIMIT                     = 0      # 0 = tất cả video. Để 1 khi chạy thử.

TIER_KEYFRAME = 2   # IndexTier.INDEX_TIER_KEYFRAME, proto/searchcore/v1/common.proto
SHARD_SIZE    = 50_000

print(f"siglip model -> {SIGLIP_LOCAL_DIR}")
print(f"out/embed    -> {OUT_ROOT} / {EMBED_ROOT}")
print(f"groups       = {GROUPS or 'tất cả'}")
print(f"upload       = {DO_UPLOAD} -> {S3_URL_BASE}")

In [ ]:
# === Tải model SigLIP (đã export ONNX) từ S3 về local ===
import os
import boto3

_bucket = os.environ["AWS_BUCKET_NAME"]                 # aic-bucket-2026
_c = boto3.client(
    "s3",
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY") or None,
    aws_secret_access_key=os.environ.get("AWS_SECRET_KEY") or None,
    region_name=os.environ["AWS_REGION"],
)


def _download_prefix(bucket, prefix, local_root):
    """Tải mọi object dưới prefix về local_root, giữ nguyên cấu trúc thư mục."""
    n = 0
    for page in _c.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            dst = os.path.join(local_root, os.path.relpath(key, prefix))
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            _c.download_file(bucket, key, dst)
            n += 1
    print(f"  {prefix} -> {local_root}: {n} file")


_download_prefix(_bucket, SIGLIP_S3_PREFIX, SIGLIP_LOCAL_DIR)

In [ ]:
import importlib
import subprocess
import sys


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except ImportError:
        return False


def _pip(*args):
    print(f"  cài: {' '.join(args)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)


def _has_onnxruntime_gpu():
    """onnxruntime (CPU) và onnxruntime-gpu cùng export module tên `onnxruntime` --
    `_has("onnxruntime")` generic ở dưới sẽ luôn True vì Kaggle base image có sẵn bản
    CPU từ trước, nên nhánh chung KHÔNG BAO GIỜ cài onnxruntime-gpu dù đó rõ ràng là ý
    định (đây chính là lý do notebook chạy CPU dù máy có GPU). Phải check thẳng
    CUDAExecutionProvider, "import được" không nói lên gì về việc có đúng bản GPU không.
    """
    try:
        import onnxruntime as ort

        return "CUDAExecutionProvider" in ort.get_available_providers()
    except ImportError:
        return False


# (tên module để import, tên package để pip). Chỉ cài cái thiếu -> rerun nhanh.
# onnxruntime-gpu xử lý riêng bên dưới, KHÔNG cho vào vòng lặp này -- xem
# _has_onnxruntime_gpu().
for mod, pkg in (
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("pyarrow", "pyarrow"),
    ("PIL", "pillow"),
    ("transformers", "transformers>=4.37,<6"),
    ("boto3", "boto3"),
):
    if _has(mod):
        print(f"  có sẵn: {mod}")
    else:
        _pip(pkg)

if _has_onnxruntime_gpu():
    print("  có sẵn: onnxruntime (CUDAExecutionProvider)")
else:
    _pip("onnxruntime-gpu")
    print("  !! Vừa cài onnxruntime-gpu -- RESTART KERNEL rồi chạy lại từ đầu.")
    print("     (nếu onnxruntime bản CPU đã bị import trong session này, cài lại")
    print("     không tự áp dụng cho tới khi restart kernel.)")


In [ ]:
def _check(ok, label, hint=""):
    print(("  OK   " if ok else "  MISS ") + label + ("" if ok else f"\n         -> {hint}"))
    return bool(ok)


problems = []

onnx_path = os.path.join(SIGLIP_LOCAL_DIR, "model.onnx")
cfg_path = os.path.join(SIGLIP_LOCAL_DIR, "preprocessor_config.json")
if not _check(os.path.exists(onnx_path), f"model.onnx = {onnx_path}",
              f"chạy lại cell tải model SigLIP — thiếu s3://{AWS_BUCKET_NAME}/{SIGLIP_S3_PREFIX}"):
    problems.append("model.onnx")
if not _check(os.path.exists(cfg_path), f"preprocessor_config.json = {cfg_path}"):
    problems.append("preprocessor_config.json")

try:
    import onnxruntime as ort

    providers = ort.get_available_providers()
    _check(True, f"onnxruntime {ort.__version__}, providers={providers}")
    _check("CUDAExecutionProvider" in providers,
           "CUDAExecutionProvider có sẵn",
           "không có GPU provider -> chạy CPU (chậm hơn nhiều với so400m)")
except ImportError:
    problems.append("onnxruntime")
    _check(False, "onnxruntime", "pip install onnxruntime-gpu")

free_gb = __import__("shutil").disk_usage("/kaggle/working").free / 2**30
_check(free_gb > 5, f"đĩa trống {free_gb:.1f} GB",
       "bật DELETE_LOCAL_AFTER_UPLOAD nếu group lớn")

print()
print("THIẾU: " + ", ".join(problems) if problems else "Preflight OK — chạy tiếp được")

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

_s3_client = None


def get_client():
    """boto3 S3 client, lazy singleton (đọc AWS_* từ env)."""
    global _s3_client
    if _s3_client is None:
        import boto3

        ak, sk = os.environ.get("AWS_ACCESS_KEY"), os.environ.get("AWS_SECRET_KEY")
        region = os.environ.get("AWS_REGION")
        _s3_client = (boto3.client("s3", aws_access_key_id=ak, aws_secret_access_key=sk,
                                   region_name=region)
                      if ak and sk else boto3.client("s3", region_name=region))
    return _s3_client


def _list_common_prefixes(prefix):
    """Các "thư mục con" trực tiếp dưới prefix trên S3 (list_objects_v2 Delimiter="/")."""
    out = []
    for page in get_client().get_paginator("list_objects_v2").paginate(
            Bucket=os.environ["AWS_BUCKET_NAME"], Prefix=prefix, Delimiter="/"):
        out.extend(p["Prefix"] for p in page.get("CommonPrefixes", []))
    return out


def _natkey(s):
    """Sort tự nhiên: L9_a đứng TRƯỚC L30_a (sort chuỗi thường thì ngược)."""
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def list_groups():
    """Tên group đã có Keyframes_{group}/ trên S3, natural sort."""
    groups = [p[len("Keyframes_"):].rstrip("/") for p in _list_common_prefixes("Keyframes_")]
    return sorted(groups, key=_natkey)


def list_videos(group):
    """Tên video (đã cắt xong keyframes) trong 1 group, natural sort."""
    prefix = f"Keyframes_{group}/keyframes/"
    names = [p[len(prefix):].rstrip("/") for p in _list_common_prefixes(prefix)]
    return sorted(names, key=_natkey)


def select_groups(groups=None, last_n=0):
    """Lọc group cần embed. groups rỗng + last_n>0 -> lấy N group cuối.

    Group không tồn tại trên S3 -> lỗi ngay, tránh chạy xong mới phát hiện gõ sai tên.
    """
    avail = list_groups()
    if not groups and last_n:
        groups = avail[-last_n:]
    if not groups:
        return avail

    unknown = [g for g in groups if g not in avail]
    if unknown:
        raise ValueError(f"Group không tồn tại trên S3: {unknown}\nCó sẵn: {avail}")
    return [g for g in avail if g in set(groups)]


def object_exists(key):
    try:
        get_client().head_object(Bucket=os.environ["AWS_BUCKET_NAME"], Key=key)
        return True
    except Exception:
        return False


def download_file(key, local_path):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    get_client().download_file(os.environ["AWS_BUCKET_NAME"], key, local_path)


def download_many(pairs, max_workers=8):
    """Tải song song [(key, local_path)]. Trả list local_path, None ở vị trí lỗi.

    Đối xứng với `upload_many` — tránh tải tuần tự (mỗi file 1 round-trip S3),
    vốn là điểm nghẽn chính khi 1 video có hàng trăm keyframe.
    """
    if not pairs:
        return []
    get_client()   # khởi tạo trước khi mở pool, tránh nhiều thread cùng tạo client

    results = [None] * len(pairs)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(download_file, k, lp): i for i, (k, lp) in enumerate(pairs)}
        for fut in as_completed(futs):
            i = futs[fut]
            try:
                fut.result()
                results[i] = pairs[i][1]
            except Exception as err:
                print(f"    ! lỗi tải {pairs[i][0]}: {err}")
    return results


def upload_file(local_path, key):
    get_client().upload_file(local_path, os.environ["AWS_BUCKET_NAME"], key)
    return key


def upload_many(pairs, max_workers=8):
    """Upload song song [(local_path, key)]. Trả list key, None ở vị trí lỗi."""
    if not pairs:
        return []
    get_client()   # khởi tạo trước khi mở pool, tránh nhiều thread cùng tạo client

    results = [None] * len(pairs)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(upload_file, lp, k): i for i, (lp, k) in enumerate(pairs)}
        for fut in as_completed(futs):
            i = futs[fut]
            try:
                results[i] = fut.result()
            except Exception as err:
                print(f"    ! lỗi upload {pairs[i][0]}: {err}")
    return results


def upload_tree(local_root, s3_prefix):
    """Upload mọi file trong local_root, key = s3_prefix + đường dẫn tương đối."""
    pairs = []
    for root, _dirs, files in os.walk(local_root):
        for fname in files:
            lp = os.path.join(root, fname)
            rel = os.path.relpath(lp, local_root).replace("\\", "/")
            pairs.append((lp, f"{s3_prefix}/{rel}"))

    keys = upload_many(pairs)
    ok = sum(k is not None for k in keys)
    print(f"    S3: {ok}/{len(pairs)} file -> {s3_prefix}/")
    return dict(zip((p[0] for p in pairs), keys))

In [ ]:
def load_siglip(local_dir, device=None):
    """Nạp SigLIP đã export ONNX (nạp 1 lần, tái dùng cho mọi video).

    Khớp `services/ingest/src/ingest/stages/embed.py::load_siglip` — cùng bundle
    (model.onnx + preprocessor_config.json), cùng cách chọn execution provider.
    """
    import onnxruntime as ort
    from transformers import SiglipImageProcessor

    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if device == "cuda" else ["CPUExecutionProvider"]
    )
    session = ort.InferenceSession(os.path.join(local_dir, "model.onnx"), providers=providers)
    image_processor = SiglipImageProcessor.from_pretrained(local_dir)
    return session, image_processor


def embed_keyframes(kf_dir, keyframes, session, image_processor, *, batch_size=BATCH_SIZE):
    """Encode ảnh keyframe (theo `frame`, file `{frame:06d}.webp` trong kf_dir)
    -> vector fp32 L2-normalized. L2-normalize đã bake vào graph ONNX, không cần
    normalize lại (xem export_siglip_onnx.py::_ImageEncoder).
    """
    from PIL import Image

    if not keyframes:
        return np.empty((0, 0), dtype=np.float32)

    vectors = None
    for start in range(0, len(keyframes), batch_size):
        batch = keyframes[start : start + batch_size]
        images = [
            Image.open(os.path.join(kf_dir, f"{kf['frame']:06d}.webp")).convert("RGB")
            for kf in batch
        ]
        pixel_values = image_processor(images=images, return_tensors="np")["pixel_values"]
        feats = session.run(["image_embeds"], {"pixel_values": pixel_values.astype(np.float32)})[0]

        if vectors is None:
            vectors = np.empty((len(keyframes), feats.shape[1]), dtype=np.float32)
        vectors[start : start + len(batch)] = feats

    return vectors

In [ ]:
def assign_scene_idx(keyframes, scenes):
    """Map mỗi keyframe -> scene_id chứa nó, theo `frame` nằm trong
    [start_frame, end_frame] của scene. Khớp
    `services/ingest/src/ingest/stages/embed.py::assign_scene_idx`.
    """
    result = []
    s = 0
    for kf in keyframes:
        while s < len(scenes) - 1 and kf["frame"] > scenes[s]["end_frame"]:
            s += 1
        result.append(scenes[s]["scene_id"])
    return result


def build_payload_rows(video_name, group, keyframes, scenes):
    """Payload cho mỗi keyframe — khớp cột trong `Payload` (proto common.proto),
    tier=KEYFRAME. `keyframe_key`/`clip_key` lấy thẳng URL S3 đã có sẵn trong
    keyframes.json/scenes.json (do notebook cắt frame ghi ra), không tính lại.

    KHÔNG có `point_id`/`video_id` int ở đây — notebook Kaggle này không có
    Postgres, việc cấp id ổn định (video_id<<32|frame, xem
    `stages/embed.py::keyframe_point_id`) để dành cho bước sau (có DB access,
    trước khi đưa vào build_index.py). Dùng `video_name` (string) để join lại.
    """
    scene_idx_by_kf = assign_scene_idx(keyframes, scenes)
    rows = []
    for kf, scene_idx in zip(keyframes, scene_idx_by_kf):
        scene = scenes[scene_idx]
        rows.append({
            "video_name": video_name,
            "group": group,
            "frame": kf["frame"],
            "keyframe_time": kf["timestamp"],
            "scene_idx": scene_idx,
            "start_sec": scene["start_time"],
            "end_sec": scene["end_time"],
            "objects": [],
            "has_ocr": False,
            "has_speech": bool(scene.get("script")),
            "keyframe_key": kf["keyframe_url"],
            "clip_key": scene.get("scene_url"),
            "tier": TIER_KEYFRAME,
        })
    return rows


def write_embed_shards(video_name, group, keyframes, scenes, vectors, out_dir,
                       *, shard_size=SHARD_SIZE):
    """Ghi vector + payload ra parquet, tối đa shard_size vector/file. Trả list path."""
    import pyarrow as pa
    import pyarrow.parquet as pq

    if len(keyframes) != len(vectors):
        raise ValueError(f"Số keyframe ({len(keyframes)}) khác số vector ({len(vectors)})")

    os.makedirs(out_dir, exist_ok=True)
    payload_rows = build_payload_rows(video_name, group, keyframes, scenes)
    dim = int(vectors.shape[1]) if len(vectors) else 0

    paths = []
    for shard_id, start in enumerate(range(0, max(len(keyframes), 1), shard_size)):
        if start >= len(keyframes):
            break
        end = min(start + shard_size, len(keyframes))
        chunk_payload = payload_rows[start:end]
        table = pa.table({
            "video_name": [p["video_name"] for p in chunk_payload],
            "group": [p["group"] for p in chunk_payload],
            "frame": [p["frame"] for p in chunk_payload],
            "keyframe_time": [p["keyframe_time"] for p in chunk_payload],
            "scene_idx": [p["scene_idx"] for p in chunk_payload],
            "start_sec": [p["start_sec"] for p in chunk_payload],
            "end_sec": [p["end_sec"] for p in chunk_payload],
            "objects": [p["objects"] for p in chunk_payload],
            "has_ocr": [p["has_ocr"] for p in chunk_payload],
            "has_speech": [p["has_speech"] for p in chunk_payload],
            "keyframe_key": [p["keyframe_key"] for p in chunk_payload],
            "clip_key": [p["clip_key"] for p in chunk_payload],
            "tier": [p["tier"] for p in chunk_payload],
            "dim": [dim] * (end - start),
            # PHẢI khai type: .tolist() trả Python float (fp64) nên PyArrow suy ra
            # list<double> -> parquet nặng GẤP ĐÔI và load_shards phải ép kiểu,
            # tốn thêm 2x bộ nhớ (4.29 GB thay vì 2.15 GB ở 500k x 1152).
            # Gộp được với shard cũ đã lỡ ghi fp64 (concat promote sang kiểu rộng hơn).
            "vector": pa.array([vectors[i].tolist() for i in range(start, end)],
                               type=pa.list_(pa.float32())),
        })
        path = os.path.join(out_dir, f"embed_{shard_id:03d}.parquet")
        pq.write_table(table, path)
        paths.append(path)
    return paths

In [ ]:
import json
import shutil


def process_video_embed(group, name, *, session, image_processor, upload=None):
    """Tải keyframes+scenes của 1 video từ S3 -> embed -> ghi + upload parquet."""
    upload = DO_UPLOAD if upload is None else upload

    kf_prefix = f"Keyframes_{group}/keyframes/{name}"
    sc_prefix = f"Keyscence_{group}/keyscence/{name}"
    kf_dir = os.path.join(OUT_ROOT, kf_prefix)
    embed_dir = os.path.join(EMBED_ROOT, f"Embeddings_{group}", "embeddings", name)

    # 1) Tải keyframes.json + scenes.json (không cần *.mp4 để embed).
    download_file(f"{kf_prefix}/keyframes.json", os.path.join(kf_dir, "keyframes.json"))
    download_file(f"{sc_prefix}/scenes.json", os.path.join(kf_dir, "scenes.json"))
    keyframes = json.load(open(os.path.join(kf_dir, "keyframes.json"), encoding="utf-8"))
    scenes = json.load(open(os.path.join(kf_dir, "scenes.json"), encoding="utf-8"))

    # 2) Tải ảnh webp cần embed — song song (giống upload_many), tránh tuần tự
    #    hàng trăm request S3 riêng lẻ.
    download_many([
        (f"{kf_prefix}/{kf['frame']:06d}.webp", os.path.join(kf_dir, f"{kf['frame']:06d}.webp"))
        for kf in keyframes
    ])

    # 3) Embed -> ghi parquet.
    vectors = embed_keyframes(kf_dir, keyframes, session, image_processor)
    paths = write_embed_shards(name, group, keyframes, scenes, vectors, embed_dir)

    # 4) Upload + dọn local.
    if upload:
        # DL-36: upload_many bắt lỗi từng file rồi trả None. Trước đây giá trị trả về
        # bị bỏ, rmtree xoá local vô điều kiện, hàm báo success -> video có shard 001
        # upload lỗi bị đánh dấu xong MÃI MÃI (OVERWRITE=False chỉ check embed_000)
        # và mất nửa keyframe. Đây là cách ">500k keyframe" âm thầm thành số khác.
        uploaded = upload_tree(embed_dir, f"Embeddings_{group}/embeddings/{name}")
        failed = [p for p, k in (uploaded or {}).items() if k is None]
        if failed:
            raise RuntimeError(
                f"{len(failed)} file upload lỗi, GIỮ local để chạy lại: {failed[:5]}")
        if DELETE_LOCAL_AFTER_UPLOAD:
            shutil.rmtree(kf_dir, ignore_errors=True)
            shutil.rmtree(embed_dir, ignore_errors=True)

    return {
        "group": group, "video": name,
        "keyframes": len(keyframes), "shards": len(paths),
        "dim": int(vectors.shape[1]) if len(vectors) else 0,
        "embed_dir": embed_dir,
    }

In [ ]:
import time

t0 = time.time()
device = "cuda" if "CUDAExecutionProvider" in __import__("onnxruntime").get_available_providers() else "cpu"
session, image_processor = load_siglip(SIGLIP_LOCAL_DIR, device)
print(f"SigLIP nạp xong ({time.time() - t0:.0f}s), device={device}")

groups_used = select_groups(GROUPS, GROUPS_LAST_N)
print(f"group chạy ({len(groups_used)}): {groups_used}")

videos = [(g, n) for g in groups_used for n in list_videos(g)]
print(f"  -> {len(videos)} video")

if NUM_SHARDS > 1:
    videos = [v for i, v in enumerate(videos) if i % NUM_SHARDS == SHARD]
    print(f"  shard {SHARD}/{NUM_SHARDS} -> {len(videos)} video")

videos = videos[: LIMIT] if LIMIT else videos
print(f"\n{len(videos)} video cần embed\n")

results, failures = [], []
for i, (group, name) in enumerate(videos, 1):
    done_key = f"Embeddings_{group}/embeddings/{name}/embed_000.parquet"
    if not OVERWRITE and object_exists(done_key):
        print(f"[{i}/{len(videos)}] {group}/{name} — đã có, bỏ qua")
        continue

    t0 = time.time()
    try:
        res = process_video_embed(group, name, session=session, image_processor=image_processor)
        res["seconds"] = round(time.time() - t0, 1)
        results.append(res)
        print(f"[{i}/{len(videos)}] {group}/{name}: {res['keyframes']} keyframe, "
              f"dim={res['dim']}, {res['shards']} shard ({res['seconds']}s)")
    except Exception as ex:
        failures.append((f"{group}/{name}", repr(ex)))
        print(f"[{i}/{len(videos)}] {group}/{name} — LỖI: {ex}")

print(f"\nxong {len(results)} video, lỗi {len(failures)}")
for name, err in failures:
    print(f"  {name}: {err}")

In [ ]:
# 1. Vector đã L2-normalize? Shape khớp keyframes?
if not results:
    print("chưa có video nào chạy xong — bỏ qua")
else:
    import pyarrow.parquet as pq

    r = results[0]
    paths = sorted(__import__("glob").glob(os.path.join(EMBED_ROOT, "**", "embed_*.parquet"),
                                           recursive=True))
    print(f"{len(paths)} shard parquet trên đĩa (đã xoá nếu DELETE_LOCAL_AFTER_UPLOAD=True)")

    print(f"\nvideo mẫu: {r['group']}/{r['video']}  "
          f"keyframe={r['keyframes']}  dim={r['dim']}  shard={r['shards']}")

In [ ]:
# 2. Xác nhận key trên S3 đúng cây (chỉ chạy khi đã upload).
if not DO_UPLOAD:
    print("DO_UPLOAD=False — chưa upload gì. Bật lên rồi chạy lại cell batch.")
elif not results:
    print("chưa có video nào chạy xong.")
else:
    r = results[0]
    prefix = f"Embeddings_{r['group']}/embeddings/{r['video']}/"
    resp = get_client().list_objects_v2(
        Bucket=os.environ["AWS_BUCKET_NAME"], Prefix=prefix, MaxKeys=10)
    objs = resp.get("Contents", [])
    print(f"{resp.get('KeyCount', 0)} object dưới {prefix}")
    for o in objs:
        print(f"  {o['Key']}  {o['Size'] / 1024:.0f} KB")

    if objs:
        import pyarrow.parquet as pq
        import io

        buf = io.BytesIO()
        get_client().download_fileobj(os.environ["AWS_BUCKET_NAME"], objs[0]["Key"], buf)
        buf.seek(0)
        table = pq.read_table(buf)
        row = table.slice(0, 1).to_pylist()[0]
        row["vector"] = row["vector"][:5] + ["..."]
        print("\nrow[0] mẫu:")
        print(json.dumps(row, ensure_ascii=False, indent=2))

In [ ]:
# ── Build FAISS snapshot (dựng sau khi embed xong, đọc lại embed_*.parquet
# từ S3) — khớp logic `services/ingest/src/ingest/build_index.py`, chạy ngay
# trong notebook để không cần máy khác. ──────────────────────────────────
SNAPSHOT_VERSION   = 1
SNAPSHOT_LOCAL_DIR = f"/kaggle/working/snapshot/v{SNAPSHOT_VERSION}"
SNAPSHOT_S3_PREFIX = f"snapshots/v{SNAPSHOT_VERSION}"
ENCODER_NAME       = "siglip-so400m-patch14-384"
HNSW_M             = 32

# Group để build index — mặc định dùng lại đúng group vừa embed ở cell batch trên.
SNAPSHOT_GROUPS = groups_used if "groups_used" in globals() else (GROUPS or list_groups())

print(f"snapshot -> {SNAPSHOT_LOCAL_DIR}  (S3: {SNAPSHOT_S3_PREFIX})")
print(f"groups   = {SNAPSHOT_GROUPS}")

In [ ]:
if not _has("faiss"):
    _pip("faiss-cpu")
else:
    print("  có sẵn: faiss")

In [ ]:
def download_embed_parquet(groups, local_root):
    """Tải mọi embed_*.parquet của các group đã cho về local_root (giữ nguyên
    key làm đường dẫn tương đối, tránh trùng tên giữa các video) — song song.
    Trả local_root.
    """
    pairs = []
    for g in groups:
        prefix = f"Embeddings_{g}/embeddings/"
        for page in get_client().get_paginator("list_objects_v2").paginate(
                Bucket=os.environ["AWS_BUCKET_NAME"], Prefix=prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if key.endswith(".parquet"):
                    pairs.append((key, os.path.join(local_root, key)))

    download_many(pairs)
    print(f"  tải {len(pairs)} file embed_*.parquet -> {local_root}")
    return local_root


EMBED_DOWNLOAD_DIR = "/kaggle/working/embed_all"
download_embed_parquet(SNAPSHOT_GROUPS, EMBED_DOWNLOAD_DIR)

In [ ]:
import glob
import hashlib
from datetime import datetime, timezone

VISUAL_FAISS = "visual.faiss"
VISUAL_REFINE = "visual.f16"
IDMAP = "idmap.npy"
PAYLOAD = "payload.parquet"
TOMBSTONE = "tombstone.bin"
MANIFEST = "manifest.json"


def load_shards(parquet_root):
    """Đọc mọi embed_*.parquet dưới parquet_root (đệ quy) -> (vectors fp32,
    payload_table không có cột vector, cùng thứ tự row). Khớp
    `services/ingest/src/ingest/build_index.py::load_shards`.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    paths = sorted(glob.glob(os.path.join(parquet_root, "**", "embed_*.parquet"), recursive=True))
    if not paths:
        raise RuntimeError(f"Không thấy embed_*.parquet nào dưới {parquet_root}")

    # promote_options="permissive": video mà MỌI scene cắt lỗi có clip_key toàn None
    # -> Arrow suy ra type `null`; gặp shard khác (type `string`) thì concat_tables
    # mặc định raise ArrowInvalid. Đã tái tạo được. Chỉ nổ ở full scale, sau khi đã
    # tải hàng chục GB parquet.
    table = pa.concat_tables([pq.read_table(p) for p in paths],
                             promote_options="permissive")

    dims = set(table.column("dim").to_pylist())   # 1 int/row, rẻ
    if len(dims) != 1:
        raise ValueError(f"Các shard có dim khác nhau: {dims}")
    dim = dims.pop()

    # ZERO-COPY. Không dùng to_pylist(): đã đo trên repo này -> peak 9.2x kích thước
    # vector (500k x 1152 = 19.7 GB thay vì 2.15 GB) vì mỗi float thành 1 PyObject
    # 24 B. Kaggle 13 GB OOM ngay ở quy mô này. Đọc thẳng buffer Arrow rồi reshape
    # cho kết quả GIỐNG HỆT (đã verify np.array_equal) với 1.0x bộ nhớ.
    col = table.column("vector").combine_chunks()
    flat = col.flatten().to_numpy(zero_copy_only=False)
    if flat.size != table.num_rows * dim:
        raise ValueError(
            f"Buffer vector {flat.size} != N*dim = {table.num_rows}*{dim} "
            "(shard có vector rỗng/null?)")
    vectors = np.ascontiguousarray(flat.reshape(table.num_rows, dim), dtype=np.float32)

    # Bất biến của toàn hệ: manifest ghi metric="cosine" nhưng index dùng
    # METRIC_INNER_PRODUCT — hai cái CHỈ tương đương khi vector đã L2-normalize.
    # Một group embed bằng bundle ONNX cũ (chưa bake L2) sẽ có norm ~15 và thống trị
    # MỌI query bất kể nội dung. Check toàn bộ, không lấy mẫu: 1 pass, ~0.5s ở 500k.
    norms = np.linalg.norm(vectors, axis=1)
    bad = np.abs(norms - 1.0) > 1e-3
    if bad.any():
        raise ValueError(
            f"{int(bad.sum())}/{len(norms)} vector KHÔNG L2-normalized "
            f"(norm min={norms.min():.4f} max={norms.max():.4f}). "
            "Group nào embed bằng bundle ONNX chưa bake L2 phải embed lại.")

    return vectors, table.drop(["vector"])


def _hash_point_id(video_name, frame):
    """Point id tất định từ (video_name, frame), không cần Postgres — khớp
    `build_index.py::_hash_point_id`.
    """
    digest = hashlib.blake2b(f"{video_name}:{frame}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "big", signed=False) & 0x7FFFFFFFFFFFFFFF


def resolve_point_ids(payload_table):
    """point_id có sẵn (pipeline có Postgres) hoặc hash từ video_name+frame
    (notebook này, không có Postgres)."""
    if "point_id" in payload_table.column_names:
        return np.asarray(payload_table.column("point_id").to_pylist(), dtype=np.int64)
    names = payload_table.column("video_name").to_pylist()
    frames = payload_table.column("frame").to_pylist()
    return np.asarray([_hash_point_id(n, f) for n, f in zip(names, frames)], dtype=np.int64)


def build_faiss_index(vectors, hnsw_m=None):
    """HNSW{hnsw_m},SQ8 metric inner-product — train + add."""
    import faiss

    hnsw_m = HNSW_M if hnsw_m is None else hnsw_m
    dim = vectors.shape[1]
    index = faiss.index_factory(dim, f"HNSW{hnsw_m},SQ8", faiss.METRIC_INNER_PRODUCT)
    index.train(vectors)
    index.add(vectors)
    return index


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_snapshot(vectors, payload_table, out_dir, *, encoder_name, version=1, hnsw_m=None,
                   groups=None, expected_count=None):
    """Ghi index/refine/idmap/payload/tombstone + manifest.json vào out_dir.
    Khớp `build_index.py::write_snapshot`.
    """
    import faiss
    import pyarrow.parquet as pq

    hnsw_m = HNSW_M if hnsw_m is None else hnsw_m
    os.makedirs(out_dir, exist_ok=True)
    n, dim = vectors.shape

    # DL-12: snapshot chỉ chứa group đã tải. Chạy notebook tuần tự từ trên xuống
    # cho SNAPSHOT_GROUPS = group của SESSION NÀY -> snapshot 3/30 group, mà count,
    # len(idmap), payload.num_rows, getsize(visual.f16) đều nhất quán VỚI NHAU nên
    # mọi validation phía core đều pass. Core serve 10% corpus mà không ai biết.
    if expected_count is not None and n != expected_count:
        raise ValueError(
            f"Số vector {n:,} != số keyframe kỳ vọng {expected_count:,}. "
            "Thiếu shard (upload lỗi?) hoặc SNAPSHOT_GROUPS không đủ group.")

    index = build_faiss_index(vectors, hnsw_m=hnsw_m)
    faiss_path = os.path.join(out_dir, VISUAL_FAISS)
    faiss.write_index(index, faiss_path)

    refine_path = os.path.join(out_dir, VISUAL_REFINE)
    np.ascontiguousarray(vectors.astype(np.float16)).tofile(refine_path)

    idmap = resolve_point_ids(payload_table)
    idmap_path = os.path.join(out_dir, IDMAP)
    np.save(idmap_path, idmap)

    payload_path = os.path.join(out_dir, PAYLOAD)
    pq.write_table(payload_table, payload_path)

    tombstone_path = os.path.join(out_dir, TOMBSTONE)
    np.zeros((n + 7) // 8, dtype=np.uint8).tofile(tombstone_path)

    # DL-10: row order = sorted(glob(...)) và KHÔNG được ghi ở đâu. Sinh lại tags.npy
    # theo VỊ TRÍ sau khi rebuild là sai âm thầm (len(tags)==count vẫn pass).
    # Ghi provenance để phát hiện, và tags PHẢI join qua idmap/point_id chứ không
    # theo vị trí.
    manifest = {
        "version": version,
        "dim": dim,
        "metric": "cosine",     # tương đương IP CHỈ vì vector đã L2-normalize
        "encoder_name": encoder_name,
        "count": n,
        "built_at": datetime.now(timezone.utc).isoformat(),
        "groups": sorted(groups) if groups else None,
        "row_order": "sorted(glob(embed_*.parquet)) — tags phải join qua idmap.point_id",
        "point_id_scheme": ("payload.point_id" if "point_id" in payload_table.column_names
                            else "blake2b64(video_name:frame)"),
        "vectors": [
            {"name": "visual", "tier": "KEYFRAME", "faiss": VISUAL_FAISS, "refine": VISUAL_REFINE}
        ],
        "checksums": {
            VISUAL_FAISS: _sha256(faiss_path),
            VISUAL_REFINE: _sha256(refine_path),
            IDMAP: _sha256(idmap_path),
            PAYLOAD: _sha256(payload_path),
            TOMBSTONE: _sha256(tombstone_path),
        },
    }
    with open(os.path.join(out_dir, MANIFEST), "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return manifest

In [ ]:
vectors, payload_table = load_shards(EMBED_DOWNLOAD_DIR)
print(f"nạp {vectors.shape[0]} vector, dim={vectors.shape[1]}")

# DL-12: đối chiếu số video CÓ TRONG SNAPSHOT với số video thật của các group đã chọn.
# Thiếu shard (upload lỗi, hoặc SNAPSHOT_GROUPS không đủ) là lỗi âm thầm: mọi con số
# trong snapshot vẫn nhất quán với nhau nên core không phát hiện được.
_in_snap = set(payload_table.column("video_name").to_pylist())
_expected = {n for g in SNAPSHOT_GROUPS for n in list_videos(g)}
_missing = sorted(_expected - _in_snap)
print(f"video: {len(_in_snap)} trong snapshot / {len(_expected)} thật trên S3")
if _missing:
    raise RuntimeError(
        f"THIẾU {len(_missing)} video trong snapshot: {_missing[:10]}"
        f"{' ...' if len(_missing) > 10 else ''}\n"
        "Chạy lại cell embed cho các video này, hoặc bỏ group khỏi SNAPSHOT_GROUPS.")

manifest = write_snapshot(vectors, payload_table, SNAPSHOT_LOCAL_DIR,
                          encoder_name=ENCODER_NAME, version=SNAPSHOT_VERSION, hnsw_m=HNSW_M,
                          groups=SNAPSHOT_GROUPS)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

if DO_UPLOAD:
    upload_tree(SNAPSHOT_LOCAL_DIR, SNAPSHOT_S3_PREFIX)

In [ ]:
# Round-trip: load lại đúng file vừa ghi, search 1 vector đã biết, kiểm
# top-1 phải trả về đúng point_id của chính vector đó (giống
# test_build_index.py::test_search_returns_correct_point_id_for_exact_match).
import faiss

_idx = faiss.read_index(os.path.join(SNAPSHOT_LOCAL_DIR, VISUAL_FAISS))
_idmap = np.load(os.path.join(SNAPSHOT_LOCAL_DIR, IDMAP))

query = vectors[:1]
_dist, _ind = _idx.search(query, 5)
print("top-5 point_id cho vector[0]:", _idmap[_ind[0]].tolist())
print("point_id thật của vector[0]:", int(_idmap[0]))

assert _idmap[_ind[0][0]] == _idmap[0], "self-match không phải top-1 — kiểm tra lại build"
print("\nOK — round trip build -> load -> search khớp")